In [3]:
"""
plot_ieeg_with_pain.py

Publication-style iEEG time series plot with pain score overlays.
Replicates the style in the reference figure: two channels from the same
electrode shaft, z-scored, stacked vertically, with pain report windows
shaded and a scale bar.

Usage
-----
    from plot_ieeg_with_pain import plot_ieeg_pain_overlay

    fig = plot_ieeg_pain_overlay(
        nwb_path   = '/path/to/sub-259_ses-01_run-IA6194BX.nwb',
        channel_a  = 'LH1',   # label in electrode 'location' column
        channel_b  = 'LH2',
        duration_min = 120,    # total window to show (e.g. whole run)
        segment    = 'middle',
        pain_scores = pain_df, # DataFrame with columns ['datetime', 'pain_score']
        pain_threshold = 1,    # shade if pain_score >= this value
    )
    fig.savefig('figure1_ieeg_pain.pdf', dpi=300, bbox_inches='tight')
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker
import pynwb
from pathlib import Path
from scipy.stats import zscore


# ── Colour palette (matches reference figure) ──────────────────────────────
TRACE_COLOR   = 'black'
SHADE_COLOR   = (0.75, 0.88, 0.95, 0.55)   # pale blue, semi-transparent
SHADE_EDGE    = (0.55, 0.75, 0.90, 0.0)
SPINE_COLOR   = '#222222'
GRID_COLOR    = '#dddddd'


# ── Helper ──────────────────────────────────────────────────────────────────
def _find_channel_index(elec_df, label: str) -> int:
    """Return the integer row index for a channel label (partial match ok)."""
    matches = elec_df.index[
        elec_df['location'].str.upper().str.startswith(label.upper())
    ].tolist()
    if not matches:
        # fallback: search label column if present
        if 'label' in elec_df.columns:
            matches = elec_df.index[
                elec_df['label'].str.upper().str.startswith(label.upper())
            ].tolist()
    if not matches:
        raise ValueError(
            f"Channel '{label}' not found. "
            f"Available: {elec_df['location'].unique()[:10].tolist()}"
        )
    return int(matches[0])


def _load_raw_segment(series, start_idx: int, end_idx: int, ch_indices: list):
    """Slice NWB data for selected channels and time range."""
    data = series.data[start_idx:end_idx, :]          # (time, all_ch)
    return data[:, ch_indices]                          # (time, selected_ch)


def _pain_windows(pain_df: pd.DataFrame,
                  run_start_dt,
                  run_end_dt,
                  pain_threshold: float,
                  half_window_min: float = 2.5) -> list:
    """
    Return list of (start_min, end_min) intervals (relative to run start)
    for pain reports >= threshold within the run.

    Parameters
    ----------
    pain_df : DataFrame with columns ['datetime', 'pain_score']
    run_start_dt : datetime-like (tz-naive)
    run_end_dt   : datetime-like (tz-naive)
    pain_threshold : shade if pain_score >= this
    half_window_min : ± minutes around each pain report to shade
    """
    if pain_df is None or len(pain_df) == 0:
        return []

    df = pain_df.copy()
    df['datetime'] = pd.to_datetime(df['datetime']).dt.tz_localize(None)

    # Filter to painful reports within run
    mask = (
        (df['pain_score'] >= pain_threshold) &
        (df['datetime'] >= run_start_dt - pd.Timedelta(minutes=half_window_min)) &
        (df['datetime'] <= run_end_dt   + pd.Timedelta(minutes=half_window_min))
    )
    df = df[mask].sort_values('datetime')

    windows = []
    for _, row in df.iterrows():
        t_min = (row['datetime'] - run_start_dt).total_seconds() / 60.0
        windows.append((t_min - half_window_min, t_min + half_window_min))
    return windows


def _merge_windows(windows: list) -> list:
    """Merge overlapping (start, end) intervals."""
    if not windows:
        return []
    windows = sorted(windows)
    merged = [windows[0]]
    for start, end in windows[1:]:
        if start <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(merged[-1][1], end))
        else:
            merged.append((start, end))
    return merged


# ── Main plot function ───────────────────────────────────────────────────────
def plot_ieeg_pain_overlay(
    nwb_path: str,
    channel_a: str,
    channel_b: str,
    duration_min: float = 120.0,
    segment: str = 'middle',
    series_name: str = 'ElectricalSeries_sEEG',
    pain_scores: pd.DataFrame = None,
    pain_threshold: float = 1.0,
    pain_half_window_min: float = 2.5,
    downsample_factor: int = 10,
    lw: float = 0.35,
    figsize: tuple = (9, 5),
    show_scale_bar: bool = True,
    scale_bar_uv: float = None,
    scale_bar_time_min: float = 2.0,
    title: str = None,
) -> plt.Figure:
    """
    Plot two iEEG channels from the same electrode shaft with pain overlays.

    Parameters
    ----------
    nwb_path : str
        Path to the NWB file (raw data).
    channel_a, channel_b : str
        Electrode labels (e.g. 'LH1', 'LH2').  Partial prefix match.
    duration_min : float
        Duration of the segment to display (minutes).
    segment : str
        'start' | 'middle' | 'end'  — which part of the run to show.
    series_name : str
        Name of the ElectricalSeries in nwb.acquisition.
    pain_scores : pd.DataFrame or None
        DataFrame with columns ['datetime', 'pain_score'].
        If None, no pain shading is drawn.
    pain_threshold : float
        Shade windows where pain_score >= this value.
    pain_half_window_min : float
        Half-width (minutes) of shaded window around each pain report.
    downsample_factor : int
        Decimate raw data by this factor before plotting (speeds rendering).
    lw : float
        Line width for traces.
    figsize : tuple
        Figure size in inches.
    show_scale_bar : bool
        Draw a scale bar instead of y-axis ticks.
    scale_bar_uv : float or None
        Height of scale bar in µV (z-scored units ≈ SD).  Auto if None.
    scale_bar_time_min : float
        Width of horizontal scale bar (minutes).
    title : str or None
        Optional figure title.

    Returns
    -------
    matplotlib.figure.Figure
    """
    # ── Load NWB ──────────────────────────────────────────────────────────
    io = pynwb.NWBHDF5IO(nwb_path, 'r')
    nwb = io.read()
    series = nwb.acquisition[series_name]

    n_samples_total, n_ch_total = series.data.shape
    sfreq = series.rate
    duration_total_min = n_samples_total / sfreq / 60.0

    # Session absolute start time (tz-naive for comparison)
    session_start = pd.Timestamp(nwb.session_start_time).tz_localize(None)

    # Channel lookup
    elec_indices = series.electrodes.data[:]
    elec_df = nwb.electrodes.to_dataframe().iloc[elec_indices].reset_index(drop=True)

    idx_a = _find_channel_index(elec_df, channel_a)
    idx_b = _find_channel_index(elec_df, channel_b)
    label_a = elec_df.loc[idx_a, 'location']
    label_b = elec_df.loc[idx_b, 'location']

    # ── Time window ───────────────────────────────────────────────────────
    time_samples = min(int(duration_min * 60 * sfreq), n_samples_total)

    if segment == 'start':
        start_idx = 0
    elif segment == 'middle':
        start_idx = (n_samples_total - time_samples) // 2
    elif segment == 'end':
        start_idx = n_samples_total - time_samples
    else:
        raise ValueError("segment must be 'start', 'middle', or 'end'")

    end_idx = start_idx + time_samples
    run_start_dt = session_start + pd.Timedelta(seconds=start_idx / sfreq)
    run_end_dt   = session_start + pd.Timedelta(seconds=end_idx   / sfreq)

    print(f"Plotting {segment} segment: "
          f"{run_start_dt.strftime('%H:%M:%S')} – {run_end_dt.strftime('%H:%M:%S')}")
    print(f"Channels: {label_a}  &  {label_b}")

    # ── Load & preprocess ─────────────────────────────────────────────────
    raw = _load_raw_segment(series, start_idx, end_idx, [idx_a, idx_b])
    io.close()

    # Z-score each channel independently
    raw_z = zscore(raw, axis=0)

    # Downsample for faster rendering
    ds = downsample_factor
    raw_z = raw_z[::ds, :]
    time_min = np.arange(raw_z.shape[0]) * ds / sfreq / 60.0  # relative to run start

    # ── Pain windows ──────────────────────────────────────────────────────
    windows = _pain_windows(
        pain_scores, run_start_dt, run_end_dt,
        pain_threshold, pain_half_window_min
    )
    windows = _merge_windows(windows)

    # ── Figure layout ─────────────────────────────────────────────────────
    # Two rows (one per channel), shared x-axis, minimal style
    fig, axes = plt.subplots(
        2, 1,
        figsize=figsize,
        sharex=True,
        gridspec_kw={'hspace': 0.08}
    )

    for ax_i, (ax, ch_idx, ch_label) in enumerate(
        zip(axes, [0, 1], [label_a, label_b])
    ):
        trace = raw_z[:, ch_idx]

        # ── Pain shading (behind trace) ──────────────────────────────────
        for (w_start, w_end) in windows:
            # Clip to plotted range
            w_start = max(w_start, 0)
            w_end   = min(w_end, duration_min)
            if w_start < w_end:
                ax.axvspan(w_start, w_end,
                           color=SHADE_COLOR[:3],
                           alpha=SHADE_COLOR[3],
                           linewidth=0,
                           zorder=1)

        # ── Trace ─────────────────────────────────────────────────────────
        ax.plot(time_min, trace,
                color=TRACE_COLOR,
                linewidth=lw,
                zorder=2)

        # ── Channel label (right side) ────────────────────────────────────
        ax.text(1.01, 0.5, ch_label,
                transform=ax.transAxes,
                va='center', ha='left',
                fontsize=8, color=SPINE_COLOR,
                fontfamily='monospace')

        # ── Spine / tick cleanup ──────────────────────────────────────────
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_visible(False)
        ax.spines['bottom'].set_color(SPINE_COLOR)
        ax.tick_params(left=False, labelleft=False)
        ax.set_yticks([])

        if ax_i == 0:  # top panel — hide x-axis spine/ticks too
            ax.spines['bottom'].set_visible(False)
            ax.tick_params(bottom=False)

    # ── X-axis (bottom panel only) ────────────────────────────────────────
    axes[-1].set_xlabel('Time (min)', fontsize=9, color=SPINE_COLOR)
    axes[-1].tick_params(labelsize=8, colors=SPINE_COLOR)
    axes[-1].xaxis.set_major_formatter(ticker.FormatStrFormatter('%g'))

    # ── Scale bars ────────────────────────────────────────────────────────
    if show_scale_bar:
        ax = axes[-1]
        xlim = ax.get_xlim()
        ylim = ax.get_ylim()

        # Auto scale bar height = 0.1 * data range (≈ 0.1 z-unit)
        if scale_bar_uv is None:
            scale_bar_uv = round(0.1 * (ylim[1] - ylim[0]), 2)

        bar_x0 = xlim[0] + 0.02 * (xlim[1] - xlim[0])
        bar_y0 = ylim[0] + 0.08 * (ylim[1] - ylim[0])

        # Vertical (amplitude) bar
        ax.plot([bar_x0, bar_x0],
                [bar_y0, bar_y0 + scale_bar_uv],
                color=SPINE_COLOR, lw=1.5, clip_on=False)
        ax.text(bar_x0 - 0.005 * (xlim[1] - xlim[0]),
                bar_y0 + scale_bar_uv / 2,
                f'{scale_bar_uv:.2g} z',
                ha='right', va='center', fontsize=7, color=SPINE_COLOR)

        # Horizontal (time) bar
        ax.plot([bar_x0, bar_x0 + scale_bar_time_min],
                [bar_y0, bar_y0],
                color=SPINE_COLOR, lw=1.5, clip_on=False)
        ax.text(bar_x0 + scale_bar_time_min / 2,
                bar_y0 - 0.06 * (ylim[1] - ylim[0]),
                f'{scale_bar_time_min:g} min',
                ha='center', va='top', fontsize=7, color=SPINE_COLOR)

    # ── Pain legend patch ─────────────────────────────────────────────────
    if windows:
        pain_patch = mpatches.Patch(
            facecolor=SHADE_COLOR[:3],
            alpha=SHADE_COLOR[3] + 0.2,
            edgecolor='none',
            label=f'Pain ≥ {pain_threshold}'
        )
        axes[0].legend(
            handles=[pain_patch],
            loc='upper right',
            fontsize=7,
            frameon=False
        )

    # ── Title ─────────────────────────────────────────────────────────────
    if title:
        axes[0].set_title(title, fontsize=10, color=SPINE_COLOR, pad=6)

    plt.tight_layout(rect=[0, 0, 0.97, 1])
    return fig


# ── Example usage ────────────────────────────────────────────────────────────
if __name__ == '__main__':

    # ── 1. Build a pain_scores DataFrame from your aligned EHR data ─────────
    #    Columns required: 'datetime' (pd.Timestamp, tz-naive) and 'pain_score'
    #
    #    Example — replace with your actual loading code:
    #
    #    pain_df = pd.read_csv('pain_scores.csv', parse_dates=['datetime'])
    #
    #    Or from your EHR pipeline:
    #    pain_df = load_session_pain_scores(subject_id='259', session_id='01')

    # Dummy example (remove in real use):
    pain_df = pd.DataFrame({
        'datetime': pd.to_datetime([
            '2023-03-15 14:35:00',
            '2023-03-15 15:10:00',
            '2023-03-15 16:02:00',
        ]),
        'pain_score': [7, 5, 8]
    })

    NWB_PATH = '/mnt/NAS/iEEG_EHR_converted/sub-259/ses-01/ieeg/sub-259_ses-01_run-IA6194BX.nwb'

    fig = plot_ieeg_pain_overlay(
        nwb_path          = NWB_PATH,
        channel_a         = 'LH1',      # ← update to real labels
        channel_b         = 'LH2',      # ← same shaft, adjacent contact
        duration_min      = 120,         # show 2-hour segment
        segment           = 'middle',
        pain_scores       = pain_df,
        pain_threshold    = 1,           # shade any non-zero pain
        pain_half_window_min = 2.5,      # ±2.5 min around each report
        downsample_factor = 20,          # reduce to ~150 Hz equivalent for plotting
        lw                = 0.4,
        figsize           = (10, 4),
        scale_bar_time_min = 10,         # 10-min horizontal bar
        title             = 'sub-259  LH1 & LH2  —  2-hr segment',
    )

    fig.savefig('ieeg_pain_overlay.pdf', dpi=300, bbox_inches='tight')
    fig.savefig('ieeg_pain_overlay.png', dpi=200, bbox_inches='tight')
    plt.show()
    print("Saved.")

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '/mnt/NAS/iEEG_EHR_converted/sub-259/ses-01/ieeg/sub-259_ses-01_run-IA6194BX.nwb', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)